# Quantum Superdense Coding Protocol

### Overview
**Superdense Coding** is a foundational quantum communication protocol. It enables Alice to transmit **2 classical bits** of information to Bob by physically sending only **1 qubit**, provided they share a pre-entangled pair of qubits ($|\Phi^+\rangle$).

#### Protocol Architecture
* **Phase 1 (Entanglement Channel):** Initialize the shared Bell state $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$ across $q_0$ (Alice) and $q_1$ (Bob).
* **Phase 2 (Alice's Encoding):** Alice applies single-qubit Pauli operations ($I, X, Z, XZ$) exclusively to $q_0$.
* **Phase 3 (Bob's Decoding):** Bob receives $q_0$ and applies a Bell-basis change ($CNOT \to H$) to convert the joint state back to the computational basis.
* **Phase 4 (Measurement):** Measure $q_0 \to c_0$ and $q_1 \to c_1$ to read the classical bits.

### 1. Setup & Dependencies
Import the required module classes from Qiskit and Qiskit Aer to build, transpile, and simulate quantum circuits.

In [1]:
#importing the necessary classes and functions from the relevant libraries and packages

from IPython.display import display
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np
from qiskit import transpile
from qiskit_aer import AerSimulator

### 2. Superdense Coding Circuit Generator

The `create_superdense_circuit` function builds the 4-phase quantum circuit dynamically based on Alice's intended 2-bit message.

| Input Message | Alice's Operation | Global Bell State |
| :--- | :--- | :--- |
| `'00'` | Identity ($I$) | $\vert\Phi^+\rangle = \frac{1}{\sqrt{2}}(\vert 00\rangle + \vert 11\rangle)$ |
| `'01'` | Pauli-$X$ | $\vert\Psi^+\rangle = \frac{1}{\sqrt{2}}(\vert 01\rangle + \vert 10\rangle)$ |
| `'10'` | Pauli-$Z$ | $\vert\Phi^-\rangle = \frac{1}{\sqrt{2}}(\vert 00\rangle - \vert 11\rangle)$ |
| `'11'` | Pauli-$Z \cdot X$ | $\vert\Psi^-\rangle = \frac{1}{\sqrt{2}}(\vert 01\rangle - \vert 10\rangle)$ |

In [7]:
def create_superdense_circuit(message: str) -> QuantumCircuit:
    """
    Constructs a Superdense Coding circuit for a given 2-bit message.
    Message options: '00', '01', '10', '11'
    """
    qc = QuantumCircuit(2, 2)
    
    # --- PHASE 1: Create Shared Bell State |Phi+> ---
    # Qubit 0: Alice | Qubit 1: Bob
    qc.h(0)
    qc.cx(0,1)
    
    qc.barrier() # Keeps stages visually separate
    
    # --- PHASE 2: Alice's Encoding (Qubit 0 only) ---
    if message == "00":
        pass  # Identity (I)
    elif message == "01":
        # Apply Pauli-X gate to q0
        qc.x(0)
    elif message == "10":
        # Apply Pauli-Z gate to q0
        qc.z(0)
    elif message == "11":
        # Apply Z followed by X (or X then Z) to q0
        qc.z(0)
        qc.x(0)
    else:
        raise ValueError("Message must be '00', '01', '10', or '11'")
        
    qc.barrier()
    
    # --- PHASE 3: Bob's Decoding (Bell-Basis Measurement) ---
    # Apply CNOT(q0, q1), then H to q0
    qc.cx(0,1)
    qc.h(0)
    
    # --- PHASE 4: Measurement ---
    # Map q0 to classical bit 0, and q1 to classical bit 1
    # Measure q0 -> c0, q1 -> c1
    qc.measure(0,0)
    qc.measure(1,1)
    
    return qc

### 3. Automated Verification & Simulation

The test harness iterates through all four 2-bit combinations (`00`, `01`, `10`, `11`), transpiles the circuits for the `AerSimulator`, and runs 1024 shots to verify deterministic classical recovery.

In [8]:
def test_all_superdense_messages():
    """
    Automates circuit generation, execution, and verification across all 4 messages.
    """
    messages = ["00", "01", "10", "11"]
    simulator = AerSimulator()
    
    print("--- Running Superdense Coding Verification ---")
    
    for msg in messages:
        # Build circuit for the current 2-bit message
        circuit = create_superdense_circuit(msg)
        
        # Transpile and execute on simulator
        compiled_circuit = transpile(circuit, simulator)
        result = simulator.run(compiled_circuit, shots=1024).result()
        counts = result.get_counts()
        
        # Output verification
        print(f"Sent: {msg}  |  Measured Counts: {counts}")



In [9]:
test_all_superdense_messages()

--- Running Superdense Coding Verification ---
Sent: 00  |  Measured Counts: {'00': 1024}
Sent: 01  |  Measured Counts: {'10': 1024}
Sent: 10  |  Measured Counts: {'01': 1024}
Sent: 11  |  Measured Counts: {'11': 1024}


### 4. Key Takeaways

* **Doubling Information Capacity:** Under Holevo's bound, a single isolated qubit can convey at most 1 bit of classical information. Pre-shared entanglement doubles this communication capacity to 2 classical bits per transmitted qubit.
* **Local Operations, Global Steering:** Alice performs operations exclusively on her local qubit ($q_0$). However, because $q_0$ is entangled with $q_1$, her local Pauli gates deterministically transform the shared global state into one of the four orthogonal Bell states.
* **Basis Transformation Decoding:** Quantum hardware cannot directly measure Bell states. Applying $CNOT(q_0, q_1)$ followed by $H(q_0)$ acts as the exact mathematical inverse of Bell-state creation, rotating the entangled states back into standard basis states ($|00\rangle, |01\rangle, |10\rangle, |11\rangle$).
* **Quantum Communication Duality:** 
  * **Superdense Coding:** Uses **1 EPR pair** + **1 physical qubit** $\rightarrow$ Transmits **2 classical bits**.
  * **Quantum Teleportation:** Uses **1 EPR pair** + **2 classical bits** $\rightarrow$ Transmits **1 quantum state**.